# 02. Построение и фильтрация графа связей

> **Краткое резюме для руководителя**: Этот ноутбук строит граф деловых связей из извлечённых данных. Каждый клиент — узел, каждая транзакция/доверенность/зарплата — ребро. После построения граф фильтруется: удаляются слабые и случайные связи, остаётся только статистически значимый «скелет» (backbone). В результате для каждого узла вычисляются расширенные метрики: количество уникальных контрагентов, концентрация оборота на топ-5, активные месяцы, и флаг «хаб».

**User Story 2**: Как аналитик, я хочу построить граф значимых связей из извлечённых данных, отфильтровав шум.

**Процесс**:
1. Загружаем Parquet-файлы из `data/` (через PySpark → Pandas)
2. Строим гетерогенный NetworkX граф (узлы: ЮЛ, ФЛ, ИП; рёбра: транзакции, доверенности, зарплаты)
3. Вычисляем метрики рёбер (доля оборота, реципрокность)
4. Выявляем общих сотрудников между компаниями
5. Применяем пайплайн фильтрации (предфильтр + disparity filter Серрано)
6. Вычисляем расширенные метрики узлов (unique_counterparty_count, top_k_concentration, active_months, hub_flag)
7. Сохраняем отфильтрованный граф как pickle

**Предпосылки**: Запустите `01_data_extraction.ipynb` для извлечения данных.

---

## Параметры фильтрации

Эти параметры контролируют, какие рёбра останутся в графе после фильтрации:
- **MIN_TX_COUNT** — минимальное количество транзакций для сохранения ребра (↑ = строже)
- **MIN_TOTAL_AMOUNT** — минимальная суммарная сумма по ребру
- **MIN_PERIODS** — минимум кварталов, в которых наблюдалась связь (↑ = только устойчивые)
- **ALPHA** — порог значимости для disparity filter Серрано (↓ = строже, оставляет только самые значимые рёбра)

In [ ]:
# =====================================================
# ПАРАМЕТРЫ ФИЛЬТРАЦИИ (можно менять)
# =====================================================

MIN_TX_COUNT = 3          # Минимальное кол-во транзакций на ребро
MIN_TOTAL_AMOUNT = 0.0    # Минимальная суммарная сумма
MIN_PERIODS = 2           # Минимум кварталов присутствия связи
ALPHA = 0.05              # Порог значимости disparity filter

## Инициализация и загрузка данных

Загружаем Parquet-файлы, сгенерированные в `01_data_extraction`:
- **nodes.parquet** — узлы (клиенты) с атрибутами: тип, ИНН, ОКВЭД, регион
- **transaction_edges.parquet** — транзакционные рёбра с суммами и датами
- **authority_edges.parquet** — связи по доверенностям
- **salary_edges.parquet** — зарплатные выплаты (ЮЛ → ФЛ)
- **hop_distances.parquet** — расстояние каждого узла от seed-компании

In [ ]:
import sys, os
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import pickle
import pandas as pd
from src import config
from src.graph_builder import (
    build_graph, compute_edge_metrics, compute_extended_metrics,
    derive_shared_employees, get_graph_stats,
)
from src.filters import apply_filter_pipeline

# На MDP SparkSession уже инициализирована платформой
try:
    spark
    print("SparkSession already available (MDP)")
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder \
        .enableHiveSupport() \
        .getOrCreate()
    print("SparkSession created")

In [ ]:
# Загрузка Parquet через PySpark (file:// для чтения с локальной ФС на MDP)
data_dir = config.DATA_DIR
spark_dir = config.SPARK_DATA_DIR

print(f'Loading data from: {data_dir}')

nodes_pdf = spark.read.parquet(f'{spark_dir}/nodes.parquet').toPandas()
tx_edges_pdf = spark.read.parquet(f'{spark_dir}/transaction_edges.parquet').toPandas()
auth_edges_pdf = spark.read.parquet(f'{spark_dir}/authority_edges.parquet').toPandas()
salary_edges_pdf = spark.read.parquet(f'{spark_dir}/salary_edges.parquet').toPandas()

# Добавляем hop_distance к узлам
hop_pdf = spark.read.parquet(f'{spark_dir}/hop_distances.parquet').toPandas()
nodes_pdf = nodes_pdf.merge(hop_pdf, on='client_uk', how='left')

print(f'Loaded: {len(nodes_pdf)} nodes, {len(tx_edges_pdf)} tx edges, '
      f'{len(auth_edges_pdf)} auth edges, {len(salary_edges_pdf)} salary edges')

## Построение графа

Создаём гетерогенный направленный граф (DiGraph) из загруженных данных:
- **Узлы**: каждый клиент (ЮЛ/ФЛ/ИП) с атрибутами — тип, имя, ИНН, ОКВЭД, регион, hop_distance
- **Рёбра транзакций**: направленные, с суммой, количеством, датами (first/last)
- **Рёбра доверенностей**: ЮЛ → ФЛ, подписание документов
- **Рёбра зарплат**: ЮЛ → ФЛ, зарплатные выплаты

После построения вычисляются **метрики рёбер**: доля оборота каждого ребра в общем обороте узла (`amount_share`), и флаг реципрокности (`is_reciprocal` — есть ли встречные платежи).

In [ ]:
G = build_graph(nodes_pdf, tx_edges_pdf, auth_edges_pdf, salary_edges_pdf)
G = compute_edge_metrics(G)

stats = get_graph_stats(G)
print('\n' + '=' * 50)
print('RAW GRAPH STATISTICS')
print('=' * 50)
print(f"Nodes: {stats['node_count']:,}")
print(f"Edges: {stats['edge_count']:,}")
print(f"Components: {stats['weakly_connected_components']}")
print(f"Density: {stats['density']:.6f}")
print(f"\nNodes by type: {stats['nodes_by_type']}")
print(f"Edges by type: {stats['edges_by_type']}")

## Общие сотрудники

Выявляем компании, у которых есть **общие сотрудники** (по зарплатным рёбрам). Если физлицо получает зарплату от двух разных ЮЛ — это сигнал аффилированности. Такие связи добавляются в граф как рёбра типа `shared_employees` с весом = количество общих сотрудников.

In [ ]:
shared_emp_df = derive_shared_employees(salary_edges_pdf)
print(f'Shared employee pairs found: {len(shared_emp_df)}')

if len(shared_emp_df) > 0:
    # Добавляем рёбра shared_employees в граф
    for _, row in shared_emp_df.iterrows():
        a, b = int(row['company_a_uk']), int(row['company_b_uk'])
        if a in G and b in G:
            G.add_edge(a, b,
                edge_type='shared_employees',
                shared_count=int(row['shared_count']),
                weight=float(row['shared_count']),
            )
            G.add_edge(b, a,
                edge_type='shared_employees',
                shared_count=int(row['shared_count']),
                weight=float(row['shared_count']),
            )
    print(f'Graph after shared employees: {G.number_of_edges()} edges')
    shared_emp_df.head(10)

## Фильтрация (предфильтр + Disparity Filter Серрано)

Двухэтапная фильтрация удаляет шум и оставляет статистически значимые связи:

1. **Предфильтр** — отсекает рёбра с малым количеством транзакций, малой суммой или короткой историей. Это убирает случайные/разовые контакты.
2. **Disparity Filter (Серрано)** — статистический тест: для каждого ребра проверяется, является ли его доля в обороте узла значимой по сравнению с равномерным распределением. Только значимые связи (p < α) остаются.

**Типичные результаты**: retention rate 30-60% рёбер, при этом сохраняется >80% общего оборота.

In [ ]:
filtered_G, filter_stats = apply_filter_pipeline(
    G,
    min_tx_count=MIN_TX_COUNT,
    min_total_amount=MIN_TOTAL_AMOUNT,
    min_periods=MIN_PERIODS,
    alpha=ALPHA,
)

print('\n' + '=' * 50)
print('FILTERING RESULTS')
print('=' * 50)
print(f"Original:       {filter_stats['original_nodes']:,} nodes, {filter_stats['original_edges']:,} edges")
print(f"After prefilter: {filter_stats['after_prefilter_nodes']:,} nodes, {filter_stats['after_prefilter_edges']:,} edges")
print(f"Final backbone: {filter_stats['final_nodes']:,} nodes, {filter_stats['final_edges']:,} edges")
print(f"Edge retention:  {filter_stats['edge_retention_rate']:.1%}")
print(f"Weight retention: {filter_stats['weight_retention_rate']:.1%}")

In [ ]:
filtered_stats = get_graph_stats(filtered_G)
print('\nFILTERED GRAPH STATISTICS')
print(f"Nodes by type: {filtered_stats['nodes_by_type']}")
print(f"Edges by type: {filtered_stats['edges_by_type']}")
print(f"Components: {filtered_stats['weakly_connected_components']}")

## Расширенные метрики узлов

После фильтрации вычисляем для каждого узла дополнительные характеристики:

| Метрика | Описание | Интерпретация |
|---------|----------|---------------|
| **unique_counterparty_count** | Кол-во уникальных контрагентов (in + out) | Много контрагентов = крупный игрок или транзитная компания |
| **top_k_concentration** | Доля оборота на топ-5 контрагентов | Близко к 1.0 = зависимость от нескольких партнёров |
| **active_months** | Кол-во уникальных месяцев с транзакциями | Мало месяцев = возможно сезонная или разовая активность |
| **hub_flag** | Является ли узел «хабом» (> 2× медианы контрагентов) | True = ключевой узел с аномально высокой связностью |

Эти метрики используются далее для edge_score (в `03_graph_analysis`) и для hub-фильтрации.

## Сохранение результатов

In [ ]:
---

## Глоссарий терминов

| Термин | Описание |
|--------|----------|
| **DiGraph** | Направленный граф — рёбра имеют направление (отправитель → получатель) |
| **Гетерогенный граф** | Граф с узлами и рёбрами разных типов (ЮЛ, ФЛ, ИП; транзакции, доверенности, зарплаты) |
| **amount_share** | Доля суммы ребра в общем исходящем обороте узла-источника |
| **is_reciprocal** | Флаг: существуют ли встречные транзакции (A→B и B→A) |
| **shared_employees** | Связь между двумя ЮЛ через общих сотрудников (физлицо получает зарплату от обоих) |
| **Disparity Filter** | Статистический метод Серрано: сохраняет только рёбра, чья доля оборота статистически значима |
| **Backbone** | «Скелет» графа после фильтрации — только значимые связи |
| **unique_counterparty_count** | Количество уникальных контрагентов узла (входящие + исходящие транзакции) |
| **top_k_concentration** | Концентрация: какая доля исходящего оборота приходится на топ-5 получателей (0–1) |
| **active_months** | Количество календарных месяцев, в которых у узла были транзакции |
| **hub_flag** | True, если узел имеет аномально много контрагентов (> 2× медианы по графу) |

---

**Следующий шаг**: Откройте `03_graph_analysis.ipynb` для кластеризации, центральности и обнаружения паттернов.

In [ ]:
# Сохраняем графы как pickle (локальная ФС)
raw_path = config.OUTPUT_GRAPH_PICKLE
filtered_path = config.OUTPUT_FILTERED_GRAPH_PICKLE

with open(raw_path, 'wb') as f:
    pickle.dump(G, f)
print(f'Raw graph saved: {raw_path}')

with open(filtered_path, 'wb') as f:
    pickle.dump(filtered_G, f)
print(f'Filtered graph saved: {filtered_path}')

# Сохраняем shared employees как Parquet (через Pandas — локальная ФС)
if len(shared_emp_df) > 0:
    se_path = config.OUTPUT_SHARED_EMPLOYEES
    shared_emp_df.to_parquet(se_path, index=False)
    print(f'Shared employees saved: {se_path}')

---

**Следующий шаг**: Откройте `03_graph_analysis.ipynb` для кластеризации, центральности и обнаружения паттернов.